In [31]:
import wikipediaapi
import requests

wiki = wikipediaapi.Wikipedia(language="en", user_agent="MyLabProject/1.0")

def get_random_titles(n=5):
    resp = requests.get(
        "https://en.wikipedia.org/w/api.php",
        params={"action": "query", "list": "random", "rnnamespace": 0, "rnlimit": n, "format": "json"},
        headers={"User-Agent": "MyLabProject/1.0"}
    )
    return [p["title"] for p in resp.json()["query"]["random"]]

titles = get_random_titles(100)
documents = []

for i, title in enumerate(titles):
    page = wiki.page(title)
    if page.exists():
        documents.append(title + " " + page.text)
        #print(f"Downloaded: {i}")

    else:
        print(f"Skipped: {title}")
print(f"\nDone! Collected {len(documents)} documents")



Done! Collected 100 documents


In [2]:
from datasets import load_dataset

print("Загружаем 10 000 статей из Википедии (это займет пару минут в первый раз)...")

dataset = load_dataset(
    "wikimedia/wikipedia",
    "20231101.en",          # актуальный дамп
    split="train[:10000]",
    trust_remote_code=False
)

documents = dataset["text"]
doc_names = dataset["title"]
print(f"Успешно загружено документов: {len(documents)}")
print(f"Пример названия первой статьи: {doc_names[0]}")

/home/typsoo/Projects/Mownit/LAB6/venv/lib64/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Загружаем 10 000 статей из Википедии (это займет пару минут в первый раз)...
Успешно загружено документов: 10000
Пример названия первой статьи: Anarchism


In [1]:
import re
from collections import Counter
import nltk
from nltk.corpus import stopwords
import numpy as np

nltk.download('stopwords')
english_stopwords = set(stopwords.words('english'))

def clean_and_tokenize(text):
    words = re.findall(r'\b[a-z]+\b', text.lower())
    cleaned_words = [w for w in words if w not in english_stopwords]
    return cleaned_words


[nltk_data] Downloading package stopwords to /home/typsoo/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [3]:
tokenized_documents = [clean_and_tokenize(doc) for doc in documents]
global_vocab = set().union(*tokenized_documents)

vocab = sorted(list(global_vocab))

vocab_index = {word: idx for idx, word in enumerate(vocab)}


tdm = np.zeros((len(vocab), len(tokenized_documents)), dtype=int)
nw = np.zeros(len(vocab), dtype=int)

for doc_idx, doc_words in enumerate(tokenized_documents):
    word_counts = Counter(doc_words)
    for word, count in word_counts.items():  
        if word in vocab_index:
            nw[vocab_index[word]] += 1
            tdm[vocab_index[word], doc_idx] = count

In [ ]:
idf = np.log((len(tokenized_documents) + 1) / (nw + 1)) + 1
tfidf_matrix = tdm * idf[:, np.newaxis]


In [52]:
query_text = "delicious food, receipt" 
k = 3


query_words = clean_and_tokenize(query_text)
query_counts = Counter(query_words)

q_vector = np.zeros(len(vocab))

for word, count in query_counts.items():
    if word in vocab_index:
        q_vector[vocab_index[word]] = count

q_vector = q_vector * idf
numerator = q_vector @ tfidf_matrix


norm_q = np.linalg.norm(q_vector)
norm_docs = np.linalg.norm(tfidf_matrix, axis=0)

if norm_q == 0: norm_q = 1e-9
norm_docs[norm_docs == 0] = 1e-9

cosine_similarities = numerator / (norm_q * norm_docs)

top_k_indices = np.argsort(cosine_similarities)[-k:][::-1]

print(f"Поиск по запросу: '{query_text}'\n" + "="*50)

for rank, doc_idx in enumerate(top_k_indices, 1):
    score = cosine_similarities[doc_idx]
    
    preview = documents[doc_idx][:150]
    
    print(f"{rank}. Документ #{doc_idx} | Совпадение: {score:.4f}")
    print(f"   Текст: {preview}...\n")

Поиск по запросу: 'delicious food, receipt'
1. Документ #7 | Совпадение: 0.0394
   Текст: Italian cuisine is a Mediterranean cuisine consisting of the ingredients, recipes, and cooking techniques developed in Italy since Roman times, and la...

2. Документ #8 | Совпадение: 0.0186
   Текст: Pasta (UK: , US: ; Italian: [ˈpasta]) is a type of food typically made from an unleavened dough of wheat flour mixed with water or eggs, and formed in...

3. Документ #6 | Совпадение: 0.0147
   Текст: Pizza is an Italian dish typically consisting of a flat base of leavened wheat-based dough topped with tomato, cheese, and other ingredients, baked at...



In [55]:
# Шаг 7: Нормализуем векторы заранее (чтобы их длина стала равна 1)
norm_docs = np.linalg.norm(tfidf_matrix, axis=0)
norm_docs[norm_docs == 0] = 1e-9 # защита от нулей
tfidf_matrix_normalized = tfidf_matrix / norm_docs

# 2. Нормализуем вектор запроса
norm_q = np.linalg.norm(q_vector)
if norm_q == 0: norm_q = 1e-9
q_vector_normalized = q_vector / norm_q

# 3. ТЕПЕРЬ ПОИСК СТАНОВИТСЯ ОДНОЙ СТРОЧКОЙ! 
# (Формула 3 из твоей методички)
cosine_similarities_fast = q_vector_normalized @ tfidf_matrix_normalized

top_k_indices = np.argsort(cosine_similarities)[-k:][::-1]

print(f"Поиск по запросу: '{query_text}'\n" + "="*50)

for rank, doc_idx in enumerate(top_k_indices, 1):
    score = cosine_similarities[doc_idx]
    
    preview = documents[doc_idx].title
    
    print(f"{rank}. Документ #{doc_idx} | Совпадение: {score:.4f}")
    print(f"   Текст: {preview}...\n")

Поиск по запросу: 'delicious food, receipt'
1. Документ #7 | Совпадение: 0.0394
   Текст: <built-in method title of str object at 0x560e37064f90>...

2. Документ #8 | Совпадение: 0.0186
   Текст: <built-in method title of str object at 0x560e37040180>...

3. Документ #6 | Совпадение: 0.0147
   Текст: <built-in method title of str object at 0x560e36e9fa10>...

4. Документ #2 | Совпадение: 0.0103
   Текст: <built-in method title of str object at 0x560e36fb3ee0>...

5. Документ #0 | Совпадение: 0.0062
   Текст: <built-in method title of str object at 0x560e36eb8200>...



In [56]:
from scipy.sparse.linalg import svds
import numpy as np

# Задаем количество "скрытых смыслов" (то самое число k)
k = 5

# Обязательно переводим матрицу в float (SVD работает только с дробными числами)
A_matrix = tfidf_matrix_normalized.astype(float)

# 1. Применяем SVD разложение. 
# Функция svds автоматически возвращает обрезанные матрицы U_k, S_k, Vt_k
U_k, S_k, Vt_k = svds(A_matrix, k=k)

# S_k возвращается как одномерный список чисел. 
# Превращаем его в диагональную матрицу D_k, как просят в формуле (4)
D_k = np.diag(S_k)

# 2. Собираем новую, "умную" матрицу A_k (без шума)
# Матричное умножение (@) трех матриц: U * D * V^T
A_k = U_k @ D_k @ Vt_k

# 3. Делаем поиск по новой мере подобия (Формула 5 из методички)
# Поскольку A_k - это новая матрица, ее столбцы снова нужно нормализовать
norm_Ak = np.linalg.norm(A_k, axis=0)
norm_Ak[norm_Ak == 0] = 1e-9
A_k_normalized = A_k / norm_Ak

# Используем наш нормализованный запрос из шага 7 для поиска по новой "умной" матрице
cosine_similarities_svd = q_vector_normalized @ A_k_normalized

# --- ВЫВОД РЕЗУЛЬТАТОВ ДЛЯ СРАВНЕНИЯ ---
top_k_svd = np.argsort(cosine_similarities_svd)[-3:][::-1]

print("\n--- РЕЗУЛЬТАТЫ С ИСПОЛЬЗОВАНИЕМ SVD (УМНЫЙ ПОИСК) ---")
for rank, doc_idx in enumerate(top_k_svd, 1):
    score = cosine_similarities_svd[doc_idx]
    print(doc_names[doc_idx])
    preview = documents[doc_idx][:150].replace('\n', ' ')
    print(f"{rank}. Документ #{doc_idx} | Совпадение: {score:.4f} | Текст: {preview}...")


--- РЕЗУЛЬТАТЫ С ИСПОЛЬЗОВАНИЕМ SVD (УМНЫЙ ПОИСК) ---
Italian cuisine
1. Документ #7 | Совпадение: 0.0355 | Текст: Italian cuisine is a Mediterranean cuisine consisting of the ingredients, recipes, and cooking techniques developed in Italy since Roman times, and la...
Pasta
2. Документ #8 | Совпадение: 0.0305 | Текст: Pasta (UK: , US: ; Italian: [ˈpasta]) is a type of food typically made from an unleavened dough of wheat flour mixed with water or eggs, and formed in...
Pizza
3. Документ #6 | Совпадение: 0.0183 | Текст: Pizza is an Italian dish typically consisting of a flat base of leavened wheat-based dough topped with tomato, cheese, and other ingredients, baked at...
